In [1]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup as bs
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from tqdm import tqdm
import pandas as pd

In [13]:
driver = webdriver.Chrome()

# 설정할 키워드와 기간 (2021년 1월 1일 ~ 2026년 3월 9일)
keyword = [~~~~~] #검색할 말 채우기

href_list_kin = []

# 1페이지부터 순회
for i in tqdm(keyword):
    # 날짜는 아래 date 관련된 내용(2개 있어요!)을 다시 수정하시면 됩니다. 언제부터 언제까지 검색해라~
    search_url =f'https://search.naver.com/search.naver?ssc=tab.kin.kqna&query={i}&sm=tab_opt&datetype=6&startdate=2023.01.01&enddate=2026.03.09&nso=so:r,p:from20230101to20260309'
    
    driver.get(search_url)
    time.sleep(1.5)

    #스크롤 다운하기(마지막까지)
    body=driver.find_element(By.CSS_SELECTOR, "body")
    while True:
        last_source=driver.page_source

        for i in range(10):
            body.send_keys(Keys.END)
            time.sleep(0.5)
        new_source = driver.page_source
        if last_source == new_source:
            break
        else:
            pass
    
    # 해당 페이지의 질문 링크들 수집
    links = driver.find_elements(By.CSS_SELECTOR, "div.sds-comps-vertical-layout.sds-comps-full-layout.FwGBOxH_GSWpSQCrzdVW > a")
    
    for link in links:
        
        href = link.get_attribute('href')
        if href:
            href_list_kin.append(href)
    
# 중복 제거
href_list_kin = list(set(href_list_kin))
print(f"최종 수집된 질문 링크 수: {len(href_list_kin)}")

최종 수집된 질문 링크 수: 189


In [15]:
kin_title = []
kin_contents =[]
kin_reply=[]

for url in tqdm(href_list_kin):    
    try:
        driver.get(url)
        
        # 1. 페이지 로딩 대기 (제목 영역이 나타날 때까지 최대 5초)
        try:
            wait = WebDriverWait(driver, 5)
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".c-heading__title, .title, h2")))
        except TimeoutException:
            print(f"로딩 시간 초과 스킵: {url}")
            # 길이를 맞추기 위해 빈값 넣고 다음 URL로
            kin_title.append("타임아웃"); kin_contents.append(""); kin_reply.append("")
            continue

        # 2. [제목 수집] - 후보군을 차례대로 확인
        title7 = ""
        title_selectors = ["div.c-heading__title-inner div.title", ".c-heading__title", "h2.title", 'div.endTitleSection']
        for sel in title_selectors:
            try:
                elem = driver.find_element(By.CSS_SELECTOR, sel)
                if elem.text.strip():
                    title7 = elem.text.strip()
                    break
            except: continue

        # 3. [본문 수집] - 질문 내용 추출
        question_content7 = ""
        content_selectors = [".c-heading__content", ".c-heading__body", "#content", "._endContentsText", 'div.questionDetail']
        for sel in content_selectors:
            try:
                elem = driver.find_element(By.CSS_SELECTOR, sel)
                if elem.text.strip():
                    # 본문 안에 제목이 중복 포함될 수 있으므로 정리
                    question_content7 = elem.text.strip()
                    break
            except: continue

        # 4. [답변 수집] - 모든 답변 텍스트 박스 통합
        answer_elements7 = driver.find_elements(By.CSS_SELECTOR, ".se-main-container, ._endContentsText, .answer-content__item, .c-answer__content, [id^='answer_']")

        temp_answers = []
        for a in answer_elements7:
            txt = a.text.strip()
            # 텍스트가 존재하고 중복되지 않은 경우만 리스트에 추가
            if txt and len(txt) > 2 and txt not in temp_answers:
                temp_answers.append(txt)

        answers_text7 = "\n[다음 답변]\n".join(temp_answers)
        
        # 5. 리스트에 저장 (데이터 정제)
        kin_title.append(title7 if title7 else "제목 없음")
        kin_contents.append(question_content7 if question_content7 else "")
        kin_reply.append(answers_text7 if answers_text7 else "")
        
    except Exception as e:
        print(f"알 수 없는 에러 ({url}): {e}")
        kin_title.append("에러"); kin_contents.append(""); kin_reply.append("")
        continue

100%|████████████████████████████████████████████████████████████████████████████████| 189/189 [10:09<00:00,  3.23s/it]


In [16]:
#수집된 것 꼭 채우세요!

['질문\n곧 있으면 은퇴할 것 같은데 노후준비...',
 '질문\n아빠가 60세에 퇴직하셨는데',
 '질문\n은퇴 후 귀촌, 서울 시니어에게 어려운 점들 뭐가 있나요?',
 '질문\n민주주의에 가까운 것',
 '질문\n노인을 위한 일자리',
 '질문\n586세대 은퇴앞둔사람들',
 '질문\n중장년 일자리',
 '질문\n도전할 직업 추천해주세요!',
 '질문\n노인장기요양등급 받았는데 갱신할때 의사소견서 또 필요한가요?',
 '질문\n시니어 초청 골든클래스, 어떤 점이 특별했나요?',
 '질문\n창업의 결과...',
 '질문\n자식이 개인사업자 지역가입자 이고 부모님이 은퇴하여 피부양자 신청하려합니다.',
 '질문\n그레이태그 ott 계정 공유 사기',
 '질문\n은퇴 후에도 계속 일하고 싶어하시는 부모님, 어떻게 지지해드려야 할까요?',
 '질문\n노후생활',
 '질문\n은퇴 생활안정자금',
 '질문\n은퇴 후에 집에서 소소하게 할 수 있는 일 있을까요?',
 '질문\n시니어 일자리 알아보려고요',
 '질문\n공무원 은퇴한사람 소득없어도 영원히 나라지원 없나요?',
 '질문\n시니어클럽 어르신 일자리에 대해',
 '질문\n웹툰제목 부탁드려요...',
 '질문\n은퇴 후 시니어 부동산창업?',
 '질문\n여성 노인분 알바',
 '질문\n12월 취업자 증가, 어느 분야가 가장 큰 혜택 봤나요?',
 '질문\n김연아 선수가 살아온 삶',
 '질문\n70대에 은퇴하신 부모님을 건강보험 피부양자로 등록했는데, 가족수당이 안나옵니다.',
 '질문\n무손사의 은퇴 후 계획, 어떤 삶을 살고 싶어하나요?',
 '질문\n50대 자격증 노후준비 가능한걸로 추천해주세요',
 '질문\n시니어클럽, 노인일자리창출지원센터',
 '질문\n김길리 선수, 차세대 에이스로 자리매김할 수 있을까요?',
 '질문\n50대에 해볼만한 재테크추천부탁.',
 '질문\n아는 친척형이 목사님인데 곧 은퇴하십니다. 은퇴선물로 무엇이 좋을까요?',
 '질문\n엄마 일자리 구할수 없을까여..',
 '질문\n

In [20]:
df_naver_kin = pd.DataFrame({
    'Title': kin_title,
    'Content': kin_contents,
    'Reply': kin_reply ,
    'Source': 'naver_kin'})

In [21]:
df_naver_kin.to_csv("senior_discomfort_total.csv", index=False, encoding='utf-8-sig')